# Minkowski Tensor Additivity by `center` Mode

Minkowski tensors satisfy an **additivity property**: for two disjoint bodies $A$ and $B$,

$$W(A \cup B) = W(A) + W(B)$$

This holds unconditionally for all tensors when `center=None` (origin as reference).
When a non-trivial `center` mode is used, the reference frame shifts before computing
position-dependent quantities, which can break additivity.

This notebook tests additivity in two ways:

- **Part 1 — No labels**: compute $W(A)$ and $W(B)$ on individual sub-meshes, compare sum to the combined mesh $W(A \cup B)$.
- **Part 2 — With labels**: pass both bodies as one mesh with a per-face label array, compare per-label sums to the combined unlabelled result.

Each part is run twice: first on a **symmetric** pair of identical boxes, then on an
**asymmetric** pair (different sizes) that reveals the full behaviour of vector tensors
under `center='centroid_mesh'`.

| `center` mode | `center_scope` | Scalars | Vectors | Rank-2 tensors |
|---|---|---|---|---|
| `None` | — | ✓ | ✓ | ✓ |
| `None` + manual global centroid pre-shift | — | ✓ | ✓ | ✓ |
| `'centroid_mesh'` | `'per_label'` | ✓ | ✗ | ✗ |
| `'centroid_mesh'` | `'global'` | ✓ | ✓ | ✓ |
| `'reference_centroid'` | — | ✓ | ✓ | ✗ |

*(Cells below confirm this table numerically.)*

In [33]:
import numpy as np
import pykarambola
from pykarambola import minkowski_tensors

print(f"pykarambola version: {pykarambola.__version__}")

pykarambola version: 0.2.0


## Helper functions

In [34]:
def box_mesh(lx=1.0, ly=1.0, lz=1.0, center=(0.0, 0.0, 0.0)):
    """Triangulated axis-aligned box with outward-facing normals.

    Parameters
    ----------
    lx, ly, lz : float
        Side lengths.
    center : (3,) array_like
        Position of the box centre.

    Returns
    -------
    verts : (8, 3) ndarray
    faces : (12, 3) ndarray
    """
    cx, cy, cz = center
    ha, hb, hc = lx / 2.0, ly / 2.0, lz / 2.0
    verts = np.array([
        [cx - ha, cy - hb, cz - hc],  # 0
        [cx + ha, cy - hb, cz - hc],  # 1
        [cx + ha, cy + hb, cz - hc],  # 2
        [cx - ha, cy + hb, cz - hc],  # 3
        [cx - ha, cy - hb, cz + hc],  # 4
        [cx + ha, cy - hb, cz + hc],  # 5
        [cx + ha, cy + hb, cz + hc],  # 6
        [cx - ha, cy + hb, cz + hc],  # 7
    ], dtype=np.float64)
    faces = np.array([
        [0, 3, 2], [0, 2, 1],  # -z face
        [4, 5, 6], [4, 6, 7],  # +z face
        [0, 1, 5], [0, 5, 4],  # -y face
        [2, 3, 7], [2, 7, 6],  # +y face
        [0, 4, 7], [0, 7, 3],  # -x face
        [1, 2, 6], [1, 6, 5],  # +x face
    ], dtype=np.int64)
    return verts, faces


SCALAR_KEYS = ['w000', 'w100', 'w200', 'w300']
VECTOR_KEYS = ['w010', 'w110', 'w210', 'w310']
TENSOR_KEYS = ['w020', 'w120', 'w220', 'w320', 'w102', 'w202']


def additivity_table(wA, wB, wAB, tol=1e-8):
    """Print whether W(A) + W(B) = W(A cup B) for each functional.

    Parameters
    ----------
    wA, wB : dict
        Results for body A and body B.
    wAB : dict
        Result for the combined mesh.
    tol : float
        Absolute tolerance for the additivity check.
    """
    header = f"  {'Key':<8}  {'Additive?':<12}  max|W(A cup B) - W(A) - W(B)|"
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for category, keys in [('Scalars', SCALAR_KEYS),
                            ('Vectors', VECTOR_KEYS),
                            ('Rank-2 tensors', TENSOR_KEYS)]:
        print(f"  [{category}]")
        for key in keys:
            if key not in wA:
                continue
            a  = np.asarray(wA[key])
            b  = np.asarray(wB[key])
            ab = np.asarray(wAB[key])
            diff = float(np.max(np.abs(ab - (a + b))))
            mark = "yes" if diff < tol else "NO"
            print(f"    {key:<8}  {mark:<12}  {diff:.3e}")


def two_box_mesh(vA, fA, vB, fB):
    """Combine two sub-meshes into one mesh with per-face labels."""
    fB_global = fB + len(vA)
    verts  = np.vstack([vA, vB])
    faces  = np.vstack([fA, fB_global])
    labels = np.array([1] * len(fA) + [2] * len(fB), dtype=np.int64)
    return verts, faces, labels

---
# Case 1: Symmetric — two identical boxes

Two identical 2×2×2 cubes:
- **Box A** (label 1): centred at `(0, 0, 0)`
- **Box B** (label 2): centred at `(20, 0, 0)`

Because both boxes are the same size and are placed symmetrically about the
global centroid `(10, 0, 0)`, their surface-area moment vectors cancel in the
combined mesh. This makes vectors **appear** additive for `centroid_mesh`,
but see Case 2 for why this is a coincidence of the symmetric setup.

In [35]:
vA_sym, fA_sym = box_mesh(lx=2.0, ly=2.0, lz=2.0, center=( 0.0, 0.0, 0.0))
vB_sym, fB_sym = box_mesh(lx=2.0, ly=2.0, lz=2.0, center=(20.0, 0.0, 0.0))
verts_sym, faces_sym, labels_sym = two_box_mesh(vA_sym, fA_sym, vB_sym, fB_sym)

print(f"Box A : {len(vA_sym):2d} verts, {len(fA_sym):2d} faces  (2x2x2, label 1, centre at  (0, 0, 0))")
print(f"Box B : {len(vB_sym):2d} verts, {len(fB_sym):2d} faces  (2x2x2, label 2, centre at (20, 0, 0))")
print(f"Total : {len(verts_sym):2d} verts, {len(faces_sym):2d} faces")

Box A :  8 verts, 12 faces  (2x2x2, label 1, centre at  (0, 0, 0))
Box B :  8 verts, 12 faces  (2x2x2, label 2, centre at (20, 0, 0))
Total : 16 verts, 24 faces


## Part 1 (symmetric): No labels

Compute $W(A)$ and $W(B)$ on individual sub-meshes, then check
$W(A) + W(B) = W(A \cup B)$.

### `center=None`

In [36]:
wA  = minkowski_tensors(vA_sym,    fA_sym,    center=None)
wB  = minkowski_tensors(vB_sym,    fB_sym,    center=None)
wAB = minkowski_tensors(verts_sym, faces_sym, center=None)

print("center=None")
additivity_table(wA, wB, wAB)

center=None
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           3.553e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           4.547e-13
    w220      yes           4.547e-13
    w320      yes           2.274e-13
    w102      yes           0.000e+00
    w202      yes           0.000e+00


### `center=None` with manual global centroid pre-shift

Compute the global centre of mass ($W_0^{10}/W_0^{00}$), shift **all** vertices
by $-c_{\text{global}}$, then call `minkowski_tensors` with `center=None`.
Because the same shift is applied everywhere, additivity is fully preserved.
This is equivalent to `center='centroid_mesh'` on the combined mesh.

In [37]:
com_sym = np.array(wAB['w010']) / wAB['w000']
print(f"Global centre of mass: {com_sym}  (midpoint of the two box centres)\n")

vA_s    = vA_sym    - com_sym
vB_s    = vB_sym    - com_sym
verts_s = verts_sym - com_sym

wA_s  = minkowski_tensors(vA_s,    fA_sym,    center=None)
wB_s  = minkowski_tensors(vB_s,    fB_sym,    center=None)
wAB_s = minkowski_tensors(verts_s, faces_sym, center=None)

print("center=None + manual global centroid pre-shift")
additivity_table(wA_s, wB_s, wAB_s)

wAB_cm = minkowski_tensors(verts_sym, faces_sym, center='centroid_mesh')
diffs = [float(np.max(np.abs(np.asarray(wAB_s[k]) - np.asarray(wAB_cm[k]))))
         for k in SCALAR_KEYS + VECTOR_KEYS + TENSOR_KEYS if k in wAB_s]
print(f"\nMax diff vs center='centroid_mesh' on combined mesh: {max(diffs):.3e}")

Global centre of mass: [10.  0.  0.]  (midpoint of the two box centres)

center=None + manual global centroid pre-shift
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           3.553e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           3.553e-15
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           2.665e-15
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           0.000e+00
    w220      yes           2.220e-15
    w320      yes           2.887e-15
    w102      yes           0.000e+00
    w202      yes           0.000e+00

Max diff vs center='centroid_mesh' on combined mesh: 2.274e-13


### `center='centroid_mesh'`

Each sub-mesh is shifted to its own centroid. In the symmetric case the two
boxes are the same size, so their surface-area moment vectors happen to cancel
in the combined mesh — vectors appear additive here, but this is a coincidence
of the symmetric geometry (see Case 2).

In [38]:
wA_cm = minkowski_tensors(vA_sym, fA_sym, center='centroid_mesh')
wB_cm = minkowski_tensors(vB_sym, fB_sym, center='centroid_mesh')
# wAB_cm already computed above

print("center='centroid_mesh'  (symmetric case)")
additivity_table(wA_cm, wB_cm, wAB_cm)

center='centroid_mesh'  (symmetric case)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           1.776e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           1.599e-14
    w210      yes           1.421e-14
    w310      yes           8.771e-15
  [Rank-2 tensors]
    w020      NO            1.600e+03
    w120      NO            1.600e+03
    w220      NO            1.257e+03
    w320      NO            8.378e+02
    w102      yes           0.000e+00
    w202      yes           0.000e+00


### `center='reference_centroid'`

In [39]:
wA_rc  = minkowski_tensors(vA_sym,    fA_sym,    center='reference_centroid')
wB_rc  = minkowski_tensors(vB_sym,    fB_sym,    center='reference_centroid')
wAB_rc = minkowski_tensors(verts_sym, faces_sym, center='reference_centroid')

print("center='reference_centroid'  (symmetric case)")
additivity_table(wA_rc, wB_rc, wAB_rc)

center='reference_centroid'  (symmetric case)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           3.553e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      NO            1.600e+03
    w120      NO            1.600e+03
    w220      NO            1.257e+03
    w320      NO            8.378e+02
    w102      yes           0.000e+00
    w202      yes           0.000e+00


## Part 2 (symmetric): With labels

In [40]:
res_none = minkowski_tensors(verts_sym, faces_sym, labels=labels_sym, center=None)
print("center=None  (with labels)")
additivity_table(res_none[1], res_none[2], wAB)

center=None  (with labels)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           3.553e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           4.547e-13
    w220      yes           4.547e-13
    w320      yes           2.274e-13
    w102      yes           0.000e+00
    w202      yes           0.000e+00


In [41]:
res_none_s = minkowski_tensors(verts_s, faces_sym, labels=labels_sym, center=None)
print("center=None + manual global pre-shift  (with labels)")
additivity_table(res_none_s[1], res_none_s[2], wAB_s)

center=None + manual global pre-shift  (with labels)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           3.553e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           3.553e-15
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           2.665e-15
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           0.000e+00
    w220      yes           2.220e-15
    w320      yes           2.887e-15
    w102      yes           0.000e+00
    w202      yes           0.000e+00


In [42]:
res_cm_pl = minkowski_tensors(
    verts_sym, faces_sym, labels=labels_sym,
    center='centroid_mesh', center_scope='per_label',
)
print("center='centroid_mesh', center_scope='per_label'  (with labels, symmetric)")
additivity_table(res_cm_pl[1], res_cm_pl[2], wAB_cm)

center='centroid_mesh', center_scope='per_label'  (with labels, symmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           1.776e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           1.599e-14
    w210      yes           1.421e-14
    w310      yes           8.771e-15
  [Rank-2 tensors]
    w020      NO            1.600e+03
    w120      NO            1.600e+03
    w220      NO            1.257e+03
    w320      NO            8.378e+02
    w102      yes           0.000e+00
    w202      yes           0.000e+00


In [43]:
res_cm_gl = minkowski_tensors(
    verts_sym, faces_sym, labels=labels_sym,
    center='centroid_mesh', center_scope='global',
)
print("center='centroid_mesh', center_scope='global'  (with labels, symmetric)")
additivity_table(res_cm_gl[1], res_cm_gl[2], wAB_cm)

center='centroid_mesh', center_scope='global'  (with labels, symmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           1.776e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           1.776e-15
    w210      yes           1.421e-14
    w310      yes           7.994e-15
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           0.000e+00
    w220      yes           4.547e-13
    w320      yes           1.998e-15
    w102      yes           0.000e+00
    w202      yes           0.000e+00


In [44]:
res_rc = minkowski_tensors(
    verts_sym, faces_sym, labels=labels_sym, center='reference_centroid',
)
print("center='reference_centroid'  (with labels, symmetric)")
additivity_table(res_rc[1], res_rc[2], wAB_rc)

center='reference_centroid'  (with labels, symmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           3.553e-15
    w100      yes           0.000e+00
    w200      yes           1.776e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      NO            1.600e+03
    w120      NO            1.600e+03
    w220      NO            1.257e+03
    w320      NO            8.378e+02
    w102      yes           0.000e+00
    w202      yes           0.000e+00


---
# Case 2: Asymmetric — two boxes of different sizes

Two boxes of **different sizes**:
- **Box A** (label 1): 2×2×2 (volume = 8) at `(0, 0, 0)`
- **Box B** (label 2): 4×4×4 (volume = 64) at `(20, 0, 0)`

The global volume-weighted centroid is no longer equidistant from the two boxes:

$$c_{\text{global}} = \frac{8 \cdot 0 + 64 \cdot 20}{8 + 64} \approx (17.78,\, 0,\, 0)$$

After this shift, Box A sits at $(-17.78, 0, 0)$ and Box B at $(2.22, 0, 0)$.
The surface-area moments of the two boxes no longer cancel, so vectors are
**not** additive for `centroid_mesh` with a per-body centroid shift.

In [45]:
vA_asy, fA_asy = box_mesh(lx=2.0, ly=2.0, lz=2.0, center=( 0.0, 0.0, 0.0))  # small
vB_asy, fB_asy = box_mesh(lx=4.0, ly=4.0, lz=4.0, center=(20.0, 0.0, 0.0))  # large
verts_asy, faces_asy, labels_asy = two_box_mesh(vA_asy, fA_asy, vB_asy, fB_asy)

wAB_asy_none = minkowski_tensors(verts_asy, faces_asy, center=None)
com_asy = np.array(wAB_asy_none['w010']) / wAB_asy_none['w000']

print(f"Box A : {len(vA_asy):2d} verts, {len(fA_asy):2d} faces  (2x2x2, vol=8,  label 1, centre at  (0, 0, 0))")
print(f"Box B : {len(vB_asy):2d} verts, {len(fB_asy):2d} faces  (4x4x4, vol=64, label 2, centre at (20, 0, 0))")
print(f"Total : {len(verts_asy):2d} verts, {len(faces_asy):2d} faces")
print(f"Global centroid: {np.round(com_asy, 4)}")

Box A :  8 verts, 12 faces  (2x2x2, vol=8,  label 1, centre at  (0, 0, 0))
Box B :  8 verts, 12 faces  (4x4x4, vol=64, label 2, centre at (20, 0, 0))
Total : 16 verts, 24 faces
Global centroid: [17.7778  0.      0.    ]


## Part 3 (asymmetric): No labels

In [46]:
wA_asy  = minkowski_tensors(vA_asy,    fA_asy,    center=None)
wB_asy  = minkowski_tensors(vB_asy,    fB_asy,    center=None)
# wAB_asy_none already computed above

print("center=None  (asymmetric)")
additivity_table(wA_asy, wB_asy, wAB_asy_none)

center=None  (asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           2.842e-14
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           1.421e-14
    w220      yes           9.095e-13
    w320      yes           6.647e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [47]:
vA_asy_s    = vA_asy    - com_asy
vB_asy_s    = vB_asy    - com_asy
verts_asy_s = verts_asy - com_asy

wA_asy_s  = minkowski_tensors(vA_asy_s,    fA_asy,    center=None)
wB_asy_s  = minkowski_tensors(vB_asy_s,    fB_asy,    center=None)
wAB_asy_s = minkowski_tensors(verts_asy_s, faces_asy, center=None)

print("center=None + manual global centroid pre-shift  (asymmetric)")
additivity_table(wA_asy_s, wB_asy_s, wAB_asy_s)

center=None + manual global centroid pre-shift  (asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           2.132e-14
    w110      yes           1.421e-14
    w210      yes           1.421e-14
    w310      yes           1.421e-14
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           9.095e-13
    w220      yes           4.547e-13
    w320      yes           3.553e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [48]:
wA_asy_cm  = minkowski_tensors(vA_asy,    fA_asy,    center='centroid_mesh')
wB_asy_cm  = minkowski_tensors(vB_asy,    fB_asy,    center='centroid_mesh')
wAB_asy_cm = minkowski_tensors(verts_asy, faces_asy, center='centroid_mesh')

print("center='centroid_mesh'  (asymmetric — vectors now non-additive)")
additivity_table(wA_asy_cm, wB_asy_cm, wAB_asy_cm)

center='centroid_mesh'  (asymmetric — vectors now non-additive)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           7.816e-14
    w110      NO            7.111e+01
    w210      NO            8.378e+01
    w310      NO            6.516e+01
  [Rank-2 tensors]
    w020      NO            2.844e+03
    w120      NO            2.686e+03
    w220      NO            2.048e+03
    w320      NO            1.345e+03
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [49]:
wA_asy_rc  = minkowski_tensors(vA_asy,    fA_asy,    center='reference_centroid')
wB_asy_rc  = minkowski_tensors(vB_asy,    fB_asy,    center='reference_centroid')
wAB_asy_rc = minkowski_tensors(verts_asy, faces_asy, center='reference_centroid')

print("center='reference_centroid'  (asymmetric)")
additivity_table(wA_asy_rc, wB_asy_rc, wAB_asy_rc)

center='reference_centroid'  (asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           2.842e-14
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      NO            2.844e+03
    w120      NO            2.560e+03
    w220      NO            1.676e+03
    w320      NO            8.378e+02
    w102      yes           0.000e+00
    w202      yes           8.882e-16


## Part 4 (asymmetric): With labels

In [50]:
res_asy_none = minkowski_tensors(verts_asy, faces_asy, labels=labels_asy, center=None)
print("center=None  (with labels, asymmetric)")
additivity_table(res_asy_none[1], res_asy_none[2], wAB_asy_none)

center=None  (with labels, asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           2.842e-14
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           1.421e-14
    w220      yes           9.095e-13
    w320      yes           6.647e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [51]:
res_asy_none_s = minkowski_tensors(verts_asy_s, faces_asy, labels=labels_asy, center=None)
print("center=None + manual global pre-shift  (with labels, asymmetric)")
additivity_table(res_asy_none_s[1], res_asy_none_s[2], wAB_asy_s)

center=None + manual global pre-shift  (with labels, asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           2.132e-14
    w110      yes           1.421e-14
    w210      yes           1.421e-14
    w310      yes           1.421e-14
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           9.095e-13
    w220      yes           4.547e-13
    w320      yes           3.553e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [52]:
res_asy_cm_pl = minkowski_tensors(
    verts_asy, faces_asy, labels=labels_asy,
    center='centroid_mesh', center_scope='per_label',
)
print("center='centroid_mesh', center_scope='per_label'  (with labels, asymmetric)")
additivity_table(res_asy_cm_pl[1], res_asy_cm_pl[2], wAB_asy_cm)

center='centroid_mesh', center_scope='per_label'  (with labels, asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           7.816e-14
    w110      NO            7.111e+01
    w210      NO            8.378e+01
    w310      NO            6.516e+01
  [Rank-2 tensors]
    w020      NO            2.844e+03
    w120      NO            2.686e+03
    w220      NO            2.048e+03
    w320      NO            1.345e+03
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [53]:
res_asy_cm_gl = minkowski_tensors(
    verts_asy, faces_asy, labels=labels_asy,
    center='centroid_mesh', center_scope='global',
)
print("center='centroid_mesh', center_scope='global'  (with labels, asymmetric)")
additivity_table(res_asy_cm_gl[1], res_asy_cm_gl[2], wAB_asy_cm)

center='centroid_mesh', center_scope='global'  (with labels, asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           2.132e-14
    w110      yes           1.421e-14
    w210      yes           1.421e-14
    w310      yes           1.421e-14
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           4.547e-13
    w220      yes           4.547e-13
    w320      yes           3.664e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [54]:
res_asy_rc = minkowski_tensors(
    verts_asy, faces_asy, labels=labels_asy, center='reference_centroid',
)
print("center='reference_centroid'  (with labels, asymmetric)")
additivity_table(res_asy_rc[1], res_asy_rc[2], wAB_asy_rc)

center='reference_centroid'  (with labels, asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           2.842e-14
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      NO            2.844e+03
    w120      NO            2.560e+03
    w220      NO            1.676e+03
    w320      NO            8.378e+02
    w102      yes           0.000e+00
    w202      yes           8.882e-16


---
## Summary

The vector columns for `centroid_mesh` differ between the symmetric and asymmetric
cases, confirming that the apparent vector additivity in the symmetric case was a
geometric coincidence (surface-area moment vectors cancelled due to equal box sizes
placed equidistantly from the global centroid).

In [55]:
TOL = 1e-8

def cat_ok(wA, wB, wAB, keys):
    diffs = [float(np.max(np.abs(
        np.asarray(wAB[k]) - np.asarray(wA[k]) - np.asarray(wB[k])
    ))) for k in keys if k in wA]
    return "yes" if all(d < TOL for d in diffs) else "NO"

rows = [
    # description, wA, wB, wAB
    # --- symmetric ---
    ("[sym, no lbl]  center=None",                    wA,             wB,             wAB          ),
    ("[sym, no lbl]  center=None + manual pre-shift", wA_s,           wB_s,           wAB_s        ),
    ("[sym, no lbl]  centroid_mesh",                  wA_cm,          wB_cm,          wAB_cm       ),
    ("[sym, no lbl]  reference_centroid",             wA_rc,          wB_rc,          wAB_rc       ),
    ("[sym, lbl]     center=None",                    res_none[1],    res_none[2],    wAB          ),
    ("[sym, lbl]     center=None + manual pre-shift", res_none_s[1],  res_none_s[2],  wAB_s        ),
    ("[sym, lbl]     centroid_mesh, scope=per_label", res_cm_pl[1],   res_cm_pl[2],   wAB_cm       ),
    ("[sym, lbl]     centroid_mesh, scope=global",    res_cm_gl[1],   res_cm_gl[2],   wAB_cm       ),
    ("[sym, lbl]     reference_centroid",             res_rc[1],      res_rc[2],      wAB_rc       ),
    # --- asymmetric ---
    ("[asy, no lbl]  center=None",                    wA_asy,         wB_asy,         wAB_asy_none ),
    ("[asy, no lbl]  center=None + manual pre-shift", wA_asy_s,       wB_asy_s,       wAB_asy_s    ),
    ("[asy, no lbl]  centroid_mesh",                  wA_asy_cm,      wB_asy_cm,      wAB_asy_cm   ),
    ("[asy, no lbl]  reference_centroid",             wA_asy_rc,      wB_asy_rc,      wAB_asy_rc   ),
    ("[asy, lbl]     center=None",                    res_asy_none[1],  res_asy_none[2],  wAB_asy_none),
    ("[asy, lbl]     center=None + manual pre-shift", res_asy_none_s[1],res_asy_none_s[2],wAB_asy_s  ),
    ("[asy, lbl]     centroid_mesh, scope=per_label", res_asy_cm_pl[1], res_asy_cm_pl[2], wAB_asy_cm ),
    ("[asy, lbl]     centroid_mesh, scope=global",    res_asy_cm_gl[1], res_asy_cm_gl[2], wAB_asy_cm ),
    ("[asy, lbl]     reference_centroid",             res_asy_rc[1],    res_asy_rc[2],    wAB_asy_rc ),
]

print(f"{'Mode':<50}  {'Scalars':<9}  {'Vectors':<9}  {'Rank-2'}")
print("-" * 83)
prev_prefix = None
for desc, a, b, ab in rows:
    prefix = desc[:5]
    if prev_prefix and prefix != prev_prefix:
        print()
    prev_prefix = prefix
    print(f"  {desc:<48}  {cat_ok(a,b,ab,SCALAR_KEYS):<9}  "
          f"{cat_ok(a,b,ab,VECTOR_KEYS):<9}  {cat_ok(a,b,ab,TENSOR_KEYS)}")

Mode                                                Scalars    Vectors    Rank-2
-----------------------------------------------------------------------------------
  [sym, no lbl]  center=None                        yes        yes        yes
  [sym, no lbl]  center=None + manual pre-shift     yes        yes        yes
  [sym, no lbl]  centroid_mesh                      yes        yes        NO
  [sym, no lbl]  reference_centroid                 yes        yes        NO
  [sym, lbl]     center=None                        yes        yes        yes
  [sym, lbl]     center=None + manual pre-shift     yes        yes        yes
  [sym, lbl]     centroid_mesh, scope=per_label     yes        yes        NO
  [sym, lbl]     centroid_mesh, scope=global        yes        yes        yes
  [sym, lbl]     reference_centroid                 yes        yes        NO

  [asy, no lbl]  center=None                        yes        yes        yes
  [asy, no lbl]  center=None + manual pre-shift     yes   

---
## Center mode reference

### Mode descriptions

| Mode | Description |
|---|---|
| `center=None` | Use the mesh origin as the reference frame. No shift is applied; all tensors are fully additive. |
| `center=None` + manual pre-shift | Shift all vertices by the global centroid *before* calling `minkowski_tensors`. Equivalent to `centroid_mesh` on the combined mesh when the same shift is applied consistently; all tensors are additive. Similar to the case tested in MinK/MinT|
| `center='centroid_mesh'`, `center_scope='per_label'` | Each body is shifted to its own volume-weighted centre of mass ($W_0^{10}/W_0^{00}$) before computing. Scalars are additive; vectors and rank-2 tensors are **not** (different bodies are shifted by different amounts). |
| `center='centroid_mesh'`, `center_scope='global'` | All bodies are shifted by the single global centre of mass of the full mesh. Equivalent to the manual pre-shift above; all tensors are additive. |
| `center='reference_centroid'` | Direct port of the C++ `--reference_centroid` flag. Vertices are **not** shifted; instead, the per-tensor Minkowski centroid ($W_x^{10}/W_x^{00}$) is subtracted when building the rank-2 tensor integrand. Scalars and vectors are additive; rank-2 tensors are **not**. **Always computed per-label regardless of `center_scope`** — no global-scope variant exists for this mode. |
| `center=<(3,) array>` | Shift all vertices by a user-supplied fixed point before computing. Additivity depends on whether the same point is used for both sub-meshes and the combined mesh. |

### Availability: C++ karambola vs pykarambola

| Feature | C++ karambola | pykarambola |
|---|---|---|
| Origin reference (`center=None`) | ✓ (default) | ✓ |
| `--reference_centroid` / `center='reference_centroid'` | ✓ (per-label when `--labels` used) | ✓ (always per-label; `center_scope` ignored) |
| `center='centroid_mesh'` (volume-weighted CoM shift) | ✗ | ✓ |
| `center_scope='per_label'` / `'global'` | ✗ | ✓ (for `centroid_mesh` only) |
| Explicit array center | ✗ | ✓ |

> **Note:** `center_scope='global'` is only meaningful for `centroid_mesh`. The `reference_centroid` mode has no global-scope variant, matching the C++ implementation.